# Projet Data Analysis — Tunis (2010–2024) — Documentation complète (Phases 1 → 4)

## 0) Contexte et objectif
- **Région** : Tunis (lat ≈ 36.81, lon ≈ 10.18)
- **Période** : **2010-01-01 → 2024-12-31** (15 ans)
- **Source** : **NASA POWER** (données météo/climat quotidiennes)
- **Objectif principal (ML)** : prédire la température maximale **du lendemain**  
  \[
  y(t) = T2M\_MAX(t+1)
  \]
- **Fichier source** : `POWER_Point_Daily_20100101_20241231_036d81N_010d18E_LST.csv`

---

## 1) Phase 1 — Acquisition + Nettoyage + Feature Engineering

### 1.1 Acquisition des données (déjà faite)
- Données extraites depuis **NASA POWER** pour Tunis et exportées en CSV.
- Période respectée : **2010–2024**.

### 1.2 Lecture robuste du fichier NASA POWER
Les fichiers NASA POWER contiennent parfois un bloc texte de métadonnées entre :
- `-BEGIN HEADER-` et `-END HEADER-`

**Action réalisée :**
- Détection automatique de `-END HEADER-` puis lecture de la table après ce bloc.

### 1.3 Construction de la colonne date
Les exports “Daily Point” contiennent fréquemment :
- `YEAR` : année
- `DOY` : day-of-year (1..365/366)

**Action réalisée :**
- Reconstruction de `date` via `YEAR + DOY` avec le format **%Y%j**.

### 1.4 Nettoyage et préparation
**Actions réalisées :**
1. **Traitement des valeurs manquantes**
   - Remplacement des sentinelles (`-999`, `-999.0`, etc.) par `NaN`.

2. **Contrôles de cohérence**
   - `RH2M` (humidité) doit être entre **0 et 100**
   - `PRECTOTCORR` (précipitations) doit être **≥ 0**
   - `WS2M` (vent) doit être **≥ 0**
   - `ALLSKY_SFC_SW_DWN` (rayonnement) doit être **≥ 0**
   - Vérification `T2M_MIN ≤ T2M_MAX` (sinon valeurs invalidées)

3. **Complétion du calendrier**
   - Re-index sur un calendrier quotidien complet **(freq = D)** de 2010 à 2024.

4. **Gestion des valeurs manquantes après contrôles**
   - Interpolation temporelle (`interpolate(method="time")`)
   - Fallback : `ffill()` puis `bfill()`.

### 1.5 Feature engineering
**Objectif** : enrichir les données pour l’analyse et la modélisation.

#### A) Variables temporelles
- `annee`, `mois`, `jour`, `jour_semaine`

#### B) Saison
- `saison ∈ {Hiver, Printemps, Ete, Automne}`  
  (sans accent pour éviter les erreurs d’encodage)

#### C) Indicateurs climatiques
- **Type_de_jour** :
  - `Pluvieux` si `PRECTOTCORR ≥ seuil_pluie` (ex : 1 mm)
  - `Ensoleillee` si `ALLSKY_SFC_SW_DWN ≥ seuil_soleil` et pas pluvieux
  - sinon `Nuageuse`
- **Vague_de_chaleur** :
  - booléen (0/1) selon seuil (ex : `T2M_MAX ≥ 35°C`)
- **Jour_de_gel** :
  - booléen (0/1) si `T2M_MIN < 0°C`

#### D) Features “ML-safe” (sans fuite)
- `T2M_MAX_lag1 = T2M_MAX(t-1)`
- `T2M_MAX_roll7 = moyenne(T2M_MAX sur 7 jours passés)`
- `PRECTOTCORR_roll7` (si dispo)

### 1.6 Exports de Phase 1
- `tunis_2010_2024_clean.csv` : données nettoyées
- `tunis_2010_2024_clean_features.csv` : données nettoyées + features

---

## 2) Phase 2 — Analyse Exploratoire (EDA)

### 2.1 Visualisation des tendances temporelles
**Graphiques produits :**
- Évolution de `T2M_MAX` (série journalière)
- Évolution de `PRECTOTCORR` (pluie)
- Évolution de `ALLSKY_SFC_SW_DWN` (rayonnement)
- Moyennes **mensuelles** (meilleure lisibilité : cycle + tendance)

### 2.2 Analyse de corrélations
- Calcul d’une **matrice de corrélation** sur les variables numériques.
- Visualisation sous forme de heatmap.

**But :**
- identifier des relations possibles (ex : température ↔ rayonnement, humidité ↔ pluie, etc.)

### 2.3 Identification des patterns
**Saisonnalité / cycles**
- Climatologie mensuelle (moyenne de `T2M_MAX` par mois sur toutes les années)

**Anomalies & événements extrêmes**
- Top 10 jours :
  - les plus chauds (max `T2M_MAX`)
  - les plus pluvieux (max `PRECTOTCORR`)
- Détection d’anomalies via **z-score** sur `T2M_MAX` (ex : |z| ≥ 2.5)

### 2.4 Analyse comparative
- Comparaison annuelle :
  - `T2M_MAX_mean` (moyenne annuelle)
  - `T2M_MAX_max` (max annuel)
  - `Rain_sum` (cumul annuel des pluies)

---

## 3) Phase 3 — Modélisation Prédictive (ML)

### 3.1 Définition de la variable cible (lendemain)
- La cible est construite comme :
  \[
  y\_next(t) = T2M\_MAX(t+1)
  \]
**Action réalisée :**
- `y_next = T2M_MAX.shift(-1)`
- suppression de la dernière ligne (car pas de “lendemain”)

### 3.2 Construction de X (variables explicatives)
**Important : éviter la fuite de données**
- Retrait de `T2M_MAX` (du jour) de X
- Retrait de `date`
- Retrait de `y_next`

### 3.3 Découpage temporel train/test
- Split **80% train / 20% test**
- Respect de l’ordre chronologique (**pas de shuffle**)

### 3.4 Méthode de validation
- **TimeSeriesSplit** (cross-validation temporelle)
- Score de tuning : `neg_mean_absolute_error`

### 3.5 Modèles implémentés
**Modèles demandés :**
1. **Régression linéaire** → `Ridge`
2. **Arbre de décision** → `DecisionTreeRegressor`
3. **Forêt aléatoire** → `RandomForestRegressor`
4. **KNN** → `KNeighborsRegressor`

### 3.6 Grid Search (optimisation)
Chaque modèle est optimisé par GridSearchCV :

- **Ridge** : `alpha`
- **Decision Tree** : `max_depth`, `min_samples_split`, `min_samples_leaf`
- **Random Forest** : `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`
- **KNN** : `n_neighbors` (k), `weights`, `p` (distance)

**Choix de k (KNN)**
- k impairs de 3 à 51 (`3..51 step 2`) pour éviter les symétries et tester plusieurs niveaux de lissage.

### 3.7 Évaluation des performances
Métriques calculées sur le test :
- **MAE** : erreur absolue moyenne
- **RMSE** : pénalise davantage les grosses erreurs
- **R²** : variance expliquée

### 3.8 Sélection du meilleur modèle
- Comparaison des modèles
- Choix du meilleur selon **MAE (test)** (et RMSE/R² en support)

### 3.9 Analyse d’importance
- Si modèle = arbre/forêt : `feature_importances_`
- Sinon : **permutation importance** (importance par permutation)

### 3.10 Sauvegarde
- Sauvegarde du pipeline complet :  
  `best_model_t2mmax_nextday_2010_2024.pkl`

---

## 4) Phase 4 — Visualisation & Communication

### 4.1 Production de graphiques
**Climat**
- T2M_MAX moyenne mensuelle (cycle + tendance)
- Comparaisons annuelles
- extrêmes (jours chauds / pluvieux)

**Machine Learning**
- Réel vs Prédit (sur test)
- Résidus dans le temps
- Histogramme des résidus

### 4.2 Interprétation des résultats
- Mise en évidence de la saisonnalité (été/hiver)
- Variabilité des précipitations (épisodes concentrés)
- Analyse de la qualité de prédiction via MAE/RMSE/R²
- Interprétation des résidus (biais, erreurs sur extrêmes)

### 4.3 Synthèse des insights actionnables (collectivité)
Exemples :
- **Anticipation chaleur** : alertes, prévention santé, gestion de la demande énergétique
- **Gestion pluie extrême** : entretien drainage, planification prévention crues locales
- **Énergie & solaire** : valorisation du potentiel solaire, meilleure planification saisonnière

---

## Documentation (références)
- NASA POWER API : https://power.larc.nasa.gov/docs/services/api/
- Pandas : https://pandas.pydata.org/docs/
- Scikit-learn : https://scikit-learn.org/stable/
